# nb_mdf_20_bronze_to_silver - generic, metadata-driven Silver processing

For every object, driven entirely by `meta.ingestion_object`, `meta.column_map` and `meta.dq_rule`:

1. **Late-arriving retry** - re-check rows in `quarantine.<entity>` that failed *only* referential rules (e.g. a work order whose asset arrived a day later).
2. **Read new Bronze rows** since the Silver watermark (`_ingested_at`). Full-snapshot objects take only the latest snapshot.
3. **Map** - rename, type (`try_cast`), derive (`transform_expr`). Unmapped source columns are reported (schema drift).
4. **De-duplicate** on the primary key, newest first (`dedupe_order_by`).
5. **Data quality** - technical + business rules from metadata. `warn` keeps and flags, `quarantine` diverts, `fail` stops the object above a threshold.
6. **De-identify PII** (hash / mask / generalise / drop) - after DQ so rules see raw values, before anything is persisted.
7. **Write** - SCD1 MERGE (soft delete for full snapshots), SCD2 MERGE, or idempotent insert-only.
8. **Log** - `meta.run_log`, `meta.dq_result`, `meta.watermark`.

In [ ]:
object_ids = ""
load_group = "all"
parent_run_id = ""
fail_on_error = True

In [ ]:
%run nb_mdf_utils

In [ ]:
def retry_late_arrivals(obj, rules, keys, run_id):
    """Quarantined rows whose ONLY failures were referential are re-checked; passing rows are merged and marked RESOLVED."""
    qname = tname("quarantine", obj["silver_table"])
    ref_rules = [r for r in rules if r["check_type"] == "referential" and r["severity"] == "quarantine"]
    if not ref_rules or not table_exists(qname):
        return 0
    q = spark.table(qname).where("_quarantine_status = 'OPEN' AND _retry_eligible = true")
    if q.limit(1).count() == 0:
        return 0
    q, flags = apply_dq_rules(q, ref_rules)
    ok = q.where(" AND ".join(flags)).drop(*flags)
    resolved = ok.select("_quarantine_id").cache()
    n = resolved.count()
    if n:
        to_merge = ok.drop("_quarantine_id", "_failed_rules", "_failed_rule_types", "_quarantine_status",
                           "_quarantined_at", "_quarantine_run_id", "_retry_eligible", "_resolved_at", "_resolved_run_id")
        # the same key can be quarantined on several days - keep only the newest version before merging
        to_merge = dedupe(to_merge, keys, obj.get("dedupe_order_by"))
        write_silver(to_merge, obj, keys, is_full_snapshot_batch=False)  # never soft-delete on a partial retry batch
        (DeltaTable.forName(spark, qname).alias("t").merge(resolved.alias("s"), "t._quarantine_id = s._quarantine_id")
         .whenMatchedUpdate(set={"_quarantine_status": "'RESOLVED'", "_resolved_at": "current_timestamp()",
                                 "_resolved_run_id": f"'{run_id}'"}).execute())
    resolved.unpersist()
    return n


def bronze_to_silver(obj, run_id):
    started = utcnow()
    oid = obj["object_id"]
    bronze = tname("bronze", obj["bronze_table"])
    keys = [k.strip() for k in obj["primary_keys"].split(",")]
    rules = get_rules(oid)
    cmap = get_column_map(oid)
    msgs = []
    try:
        if not table_exists(bronze):
            log_run(run_id, parent_run_id, oid, "silver", "SKIPPED", started, message="bronze table not found")
            return "SKIPPED"

        # 1. late-arriving retry
        resolved = retry_late_arrivals(obj, rules, keys, run_id)
        if resolved:
            msgs.append(f"LATE-ARRIVING: {resolved} quarantined row(s) resolved and merged")

        # 2. new bronze rows
        wm = get_watermark(oid, "silver")
        b = spark.table(bronze)
        if wm:
            b = b.where(F.col("_ingested_at") > F.lit(wm).cast("timestamp"))
        # watermark = everything read in this run (taken BEFORE the full-snapshot filter); stored with its UTC
        # offset so it round-trips exactly, including across daylight-saving changes
        new_wm = b.agg(F.date_format(F.max("_ingested_at"), "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX")).first()[0]
        if obj["load_type"] == "full":
            latest = b.agg(F.max("_load_date")).first()[0]
            b = b.where(F.col("_load_date") == latest)
        b = b.cache()
        rows_read = b.count()
        if rows_read == 0:
            b.unpersist()
            log_run(run_id, parent_run_id, oid, "silver", "SUCCESS" if resolved else "SKIPPED", started,
                    rows_written=resolved, message="; ".join(msgs) or "no new bronze rows")
            return "SUCCESS" if resolved else "SKIPPED"

        # 3. map
        df, unmapped, warns = map_columns(b, cmap)
        if unmapped:
            msgs.append(f"UNMAPPED source columns (update meta.column_map to carry them): {', '.join(unmapped)}")
        msgs += warns

        # 4. de-duplicate
        df = dedupe(df, keys, obj.get("dedupe_order_by")).cache()
        rows_after = df.count()
        dups = rows_read - rows_after

        # 5. data quality
        df, flags = apply_dq_rules(df, rules)
        df = df.cache()
        results, _ = summarise_dq(df, rules, run_id, oid)
        write_dq_results(results)
        hard_fail = [r[2] for r in results if r[10] == "FAILED"]
        if hard_fail:
            raise Exception(f"DQ fail-severity rule(s) breached threshold: {hard_fail}")

        sev = {_flag(r): r for r in rules}
        q_flags = [f for f in flags if sev[f]["severity"] == "quarantine"]
        w_flags = [f for f in flags if sev[f]["severity"] in ("warn", "fail")]

        def failed_array(fl, attr):
            if not fl:
                return F.array().cast("array<string>")
            return F.filter(F.array(*[F.when(~F.col(f), F.lit(sev[f][attr])) for f in fl]), lambda x: x.isNotNull())

        is_q = F.lit(False)
        for f in q_flags:
            is_q = is_q | ~F.col(f)
        non_ref_q = [f for f in q_flags if sev[f]["check_type"] != "referential"]
        retry_ok = F.lit(True)
        for f in non_ref_q:
            retry_ok = retry_ok & F.col(f)
        df = (df.withColumn("_is_quarantined", is_q)
                .withColumn("_dq_warnings", failed_array(w_flags, "rule_id"))
                .withColumn("_failed_rules", failed_array(q_flags, "rule_id"))
                .withColumn("_failed_rule_types", failed_array(q_flags, "check_type"))
                .withColumn("_retry_eligible", retry_ok)
                .drop(*flags))

        # 6. PII de-identification (before anything is persisted)
        df = apply_pii(df, cmap)

        # 7a. quarantine
        qdf = (df.where("_is_quarantined")
                 .drop("_is_quarantined")
                 .withColumn("_quarantine_id", F.expr("uuid()"))
                 .withColumn("_quarantine_status", F.lit("OPEN"))
                 .withColumn("_quarantined_at", F.current_timestamp())
                 .withColumn("_quarantine_run_id", F.lit(run_id))
                 .withColumn("_resolved_at", F.lit(None).cast("timestamp"))
                 .withColumn("_resolved_run_id", F.lit(None).cast("string")))
        n_q = qdf.count()
        if n_q:
            qdf.write.format("delta").mode("append").option("mergeSchema", "true") \
               .saveAsTable(tname("quarantine", obj["silver_table"]))

        # 7b. silver
        good = df.where("NOT _is_quarantined").drop("_is_quarantined", "_failed_rules", "_failed_rule_types",
                                                     "_retry_eligible")
        metrics = write_silver(good, obj, keys, snapshot_keys=df.select(*keys))
        written = metrics.get("numTargetRowsInserted", 0) + metrics.get("numTargetRowsUpdated", 0)
        msgs.append(f"merge metrics {metrics}")
        warn_rules = [f"{r[2]}={r[8]}" for r in results if r[10] == "WARN"]
        if warn_rules:
            msgs.append(f"DQ warnings: {', '.join(warn_rules)}")

        set_watermark(oid, "silver", new_wm, run_id)
        df.unpersist(); b.unpersist()
        log_run(run_id, parent_run_id, oid, "silver", "SUCCESS", started, rows_read=rows_read, rows_written=written,
                rows_quarantined=n_q, rows_deduplicated=dups, message="; ".join(msgs))
        return "SUCCESS"
    except Exception as e:
        log_run(run_id, parent_run_id, oid, "silver", "FAILED", started,
                message=f"{type(e).__name__}: {str(e)[:1500]}; " + "; ".join(msgs))
        traceback.print_exc()
        return "FAILED"

In [ ]:
run_id = new_run_id("silver")
objects = get_objects(object_ids, load_group)   # ordered by priority: reference data before transactions
print(f"{len(objects)} object(s) to process - run {run_id}")
results = {o["object_id"]: bronze_to_silver(o, run_id) for o in objects}
display(spark.table(tname("meta", "run_log")).where(F.col("run_id") == run_id)
        .select("object_id", "status", "rows_read", "rows_written", "rows_quarantined", "rows_deduplicated", "message"))

In [ ]:
display(spark.table(tname("meta", "dq_result")).where(F.col("run_id") == run_id).where("outcome <> 'PASS'")
        .select("object_id", "rule_id", "rule_name", "rule_category", "severity", "rows_failed", "failed_pct", "outcome"))

In [ ]:
failed = [k for k, v in results.items() if v == "FAILED"]
if failed and fail_on_error:
    raise Exception(f"Silver processing failed for: {failed}. See meta.run_log for run {run_id}")
notebookutils.notebook.exit(json.dumps({"run_id": run_id, "results": results, "failed": failed}))